In [1]:
import sys
from pathlib import Path

# 1. Определяем корень проекта
# (подбираем количество .parent, чтобы попасть в max_projects)
project_root = Path.cwd().parent

# 2. Добавляем корень в пути поиска модулей
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 3. Проверяем, что путь добавлен
print(f"✅ Project root: {project_root}")
print(f"✅ Sys path: {sys.path[:3]}...")

✅ Project root: d:\Pytnon_scripts\start_vector
✅ Sys path: ['d:\\Pytnon_scripts\\start_vector', 'C:\\Users\\user\\AppData\\Local\\Programs\\Python\\Python313\\python313.zip', 'C:\\Users\\user\\AppData\\Local\\Programs\\Python\\Python313\\DLLs']...


In [2]:
import logging
import pandas as pd
import aiohttp
import asyncio
from datetime import datetime, timedelta

from src_oop.core.scraper import HTTPClient
from src_oop.core.utils_general import load_api_tokens

In [100]:
logger = logging.getLogger(__name__)

async def get_promotions(session: aiohttp.ClientSession, start_date_time: str = None, end_date_time: str = None, all_promo: bool = False, account: str = None, api_key: str = None)->list:
    """Функция для получения данных об акциях на ВБ"""
    # Обработка дат
    if start_date_time is None:
        start_date_time = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%dT%H:%M:%SZ")
    if end_date_time is None:
        end_date_time = (datetime.now() + timedelta(days=28)).strftime("%Y-%m-%dT%H:%M:%SZ")
    # URL
    url = "https://dp-calendar-api.wildberries.ru/api/v1/calendar/promotions"
    
    # Создаем клиент для соединения
    client = HTTPClient(
        session=session,
        api_key=api_key,
        timeout=30
    )

    all_data = []

    # для отслеживания не пустых ответов со стороны ВБ
    limit = 1000
    offset = 0

    while True:
        params = {
           "startDateTime": start_date_time,
           "endDateTime": end_date_time,
           "allPromo": str(all_promo).lower(),
           "limit": limit,
           "offset": offset,
        }

        # Отправляем асинхронный запрос
        data = await client.get(url, params=params, delay = 0.6)
        logger.info(f"Запрошена информация по акциям для ЛК {account}")

        # Если данные пустые выходим из цикла
        if data is None:
            logger.error(f"Данные для ЛК {account} не получены")
            break

        # Безопасное извлечение данных
        # Структура ответа WB обычно: {"data": {"promotions": [...]}}
        inner_data = data.get("data")
        if not inner_data:
            break
            
        promo_list = inner_data.get("promotions", [])
        
        if not promo_list:
            logger.info(f"Больше акций не найдено для {account}")
            break

        for promo in promo_list:
            promo["account"] = account
            all_data.append(promo) # Используем append для добавления словаря целиком

        logger.info(f"[{account}] Получено {len(promo_list)} записей (offset: {offset})")

        # 4. Проверяем, нужно ли идти на следующую страницу
        if len(promo_list) < limit:
            break
            
        offset += len(promo_list)

    logger.info(f"🏁 Всего собрано {len(all_data)} записей для {account}")
    return all_data


async def fetch_get_promotions(tokens: dict):
    """Функция для получения данных обо всех акциях на ВБ по доступным ЛК"""
    # Асинхронно обрабатываем все аккаунты и токены
    async with aiohttp.ClientSession() as session:
            tasks = [
                get_promotions(
                    session=session, 
                    account=account, 
                    api_key=token
                ) for account, token in tokens.items()
            ]
            results = await asyncio.gather(*tasks)
            flat_results = [item for sublist in results for item in sublist]
            return flat_results 

In [101]:
tokens = load_api_tokens()
promo_info = await fetch_get_promotions(tokens)
promo_info

[{'id': 2231,
  'name': 'Скидки расцвели: максимальный бустинг (автоматические скидки)',
  'startDateTime': '2026-03-31T21:00:00Z',
  'endDateTime': '2026-04-07T20:59:59Z',
  'type': 'auto',
  'account': 'Даниелян'},
 {'id': 2225,
  'name': 'Скидки расцвели: хиты (автоматические скидки)',
  'startDateTime': '2026-03-27T21:00:00Z',
  'endDateTime': '2026-04-08T20:59:59Z',
  'type': 'auto',
  'account': 'Даниелян'},
 {'id': 2217,
  'name': 'Распродажа бытовой химии и товаров для уборки - автоматические скидки',
  'startDateTime': '2026-03-25T21:00:00Z',
  'endDateTime': '2026-04-02T20:59:59Z',
  'type': 'auto',
  'account': 'Даниелян'},
 {'id': 2213,
  'name': 'Скидки расцвели: заказы на максимум - автоматические скидки',
  'startDateTime': '2026-03-24T21:00:00Z',
  'endDateTime': '2026-03-31T20:59:59Z',
  'type': 'auto',
  'account': 'Даниелян'},
 {'id': 2205,
  'name': 'Повышаем заказы - автоматические скидки',
  'startDateTime': '2026-03-20T21:00:00Z',
  'endDateTime': '2026-04-07T20:

In [74]:
tokens = load_api_tokens()
try:
    promo_info = await fetch_get_promotions(tokens)
    promo_ids = {}
    for promo in promo_info:
        account = promo.get("account")
        if account not in promo_ids:
            promo_ids[account] = set() # Создаем множество только если его еще нет
        
        p_id = promo.get("id")
        if p_id:
            promo_ids[account].add(p_id)
except Exception as e:
    logger.info(f"Не удалось извлечь данные об id {e}")


In [95]:
async def process_extract_promo_ids()->dict:
    # Извлекаем данные по id рекламных кампаний, чтобы запросить по ним данные об участвующих товарах
    tokens = load_api_tokens()
    try:
        promo_info = await fetch_get_promotions(tokens)
        promo_ids = {}
        for promo in promo_info:
            account = promo.get("account")
            if account not in promo_ids:
                promo_ids[account] = set() # Создаем множество только если его еще нет
            
            p_id = promo.get("id")
            if p_id:
                promo_ids[account].add(p_id)
    except Exception as e:
        logger.info(f"Не удалось извлечь данные об id {e}")
    return promo_ids

In [96]:
promo_ids = await process_extract_promo_ids()
promo_ids

{'Даниелян': {1787,
  2040,
  2053,
  2113,
  2114,
  2117,
  2119,
  2121,
  2125,
  2126,
  2142,
  2178,
  2180,
  2184,
  2185,
  2186,
  2196,
  2197,
  2199,
  2200,
  2202,
  2203,
  2205,
  2206,
  2212,
  2213,
  2214,
  2216,
  2217,
  2218,
  2222,
  2224,
  2225,
  2226,
  2228,
  2231,
  2232,
  2234,
  2235},
 'Вектор': {1787,
  2040,
  2053,
  2113,
  2114,
  2117,
  2119,
  2121,
  2125,
  2126,
  2142,
  2178,
  2180,
  2184,
  2185,
  2186,
  2196,
  2197,
  2199,
  2200,
  2202,
  2203,
  2206,
  2212,
  2213,
  2214,
  2216,
  2217,
  2218,
  2222,
  2224,
  2225,
  2226,
  2228,
  2231,
  2232,
  2234,
  2235},
 'Оганесян': {1787,
  2040,
  2053,
  2113,
  2114,
  2117,
  2119,
  2121,
  2125,
  2126,
  2142,
  2178,
  2180,
  2184,
  2185,
  2186,
  2196,
  2197,
  2199,
  2200,
  2202,
  2203,
  2205,
  2206,
  2212,
  2213,
  2214,
  2216,
  2217,
  2218,
  2222,
  2224,
  2225,
  2226,
  2228,
  2231,
  2232,
  2234,
  2235},
 'Хачатрян': {1787,
  2040,
  2053,

In [97]:
async def get_nomenclatures_promo(session: aiohttp.ClientSession, promo_id: int = None, account: str = None, api_key: str = None):
    url = "https://dp-calendar-api.wildberries.ru/api/v1/calendar/promotions/nomenclatures"

    # Создаем клиент для соединения
    client = HTTPClient(
        session=session,
        api_key=api_key,
        timeout=30
    )

    all_data = []

    # для отслеживания не пустых ответов со стороны ВБ
    limit = 1000
    offset = 0

    while True:
        params = {
           "promotionID": promo_id,
           "inAction": str(True).lower(),
           "limit": limit,
           "offset": offset,
        }

        # Отправляем асинхронный запрос
        data = await client.get(url, params=params, delay = 0.6)
        logger.info(f"Запрошена информация по акциям для ЛК {account}")

        # Если данные пустые выходим из цикла
        if data is None:
            logger.error(f"Данные для ЛК {account} не получены")
            break

        # Безопасное извлечение данных
        # Структура ответа WB обычно: {"data": {"promotions": [...]}}
        inner_data = data.get("data")
        if not inner_data:
            break
            
        nomenclatures_list = inner_data.get("nomenclatures", [])
        
        if not nomenclatures_list:
            logger.info(f"Больше товаров для акции {promo_id} не найдено для {account}")
            break

        for promo in nomenclatures_list:
            promo["account"] = account
            all_data.append(promo) # Используем append для добавления словаря целиком

        logger.info(f"[{account}] Получено {len(nomenclatures_list)} записей (offset: {offset})")

        # 4. Проверяем, нужно ли идти на следующую страницу
        if len(nomenclatures_list) < limit:
            break
            
        offset += len(nomenclatures_list)

    logger.info(f"🏁 Всего собрано {len(all_data)} записей для {account}")
    return all_data

async def fetch_nomenclatures_promo(tokens: dict):
    """Асинхронная функция для получения информации о рекламных кампаниях для всех аккаунтов и токенов."""
    tokens = load_api_tokens()
    # Асинхронно обрабатываем все аккаунты и токены
    async with aiohttp.ClientSession() as session:
            tasks = []
            # Создаем задачи для каждого аккаунта и токена
            for account, ids in promo_ids.items():
                token = tokens.get(account)
                for id in ids:
                    task = get_nomenclatures_promo(session, id, account, token)
                    tasks.append(task)
            # Ожидаем завершения всех задач и собираем результаты
            results = await asyncio.gather(*tasks)
            return results

In [99]:
tokens = load_api_tokens()
res = await fetch_nomenclatures_promo(tokens)

CancelledError: 

In [73]:
res

[]